In [3]:
import importlib
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter('ignore')
project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root / 'src'))

from func_data_build import build_dataset
from func_gibbs.gibbs_notebook_utils import (
    display_hmc_posterior_prior,
    display_hmc_results,
    format_sample_window,
    prepare_gibbs_sample,
    save_idata_map,
)
from func_gibbs.gibbs_wrappers import draws_to_idata

seed = 0
n_iter = 12000
burn = 4000
thin = 5

import func_gibbs.gibbs_wrappers as gibbs_module
from func_gibbs.gibbs_wrappers import run_hsa_dynamic

prior_specs = {
    'alpha': (0.5, 0.2),
    'kappa': (0.1, 0.2),
    'theta': (0.1, 0.2),
    'phi_1': (0.7, 0.2),
    'rho_1': (0.2, 0.2),
    'rho_2': (0.2, 0.2),
}

data_dir = project_root / "data"
idat_dir = project_root / "results" / "idata"
tex_root_dir = project_root / "results" / "tex" / "table"
base_fig_dir = project_root / "results" / "fig"
tex_dir = tex_root_dir / 'gibbs_hsa_dynamic'

data_origin = build_dataset(data_dir)

inflation_specs = {
    'ppi': {
        'pi': 'pi_ppi',
        'pi_prev': 'pi_ppi_prev',
        'pi_expect': 'Epi_spf_gdp',
    },
    'cpi': {
        'pi': 'pi_cpi',
        'pi_prev': 'pi_cpi_prev',
        'pi_expect': 'Epi_spf_cpi',
    },
}

x_specs = {
    'unemp_gap': 'unemp_gap',
    'output_gap_BN': 'output_gap_BN',
    'markup_BN_inv': 'markup_BN_inv',
    'markup_inv': 'markup_inv',
}


In [4]:
period_label = '1982_2012'
period_title = '1982-2012'
n_name = 'gustavo'
n_col = 'N_Gustavo'
data = data_origin.loc['1982-01-01':'2012-12-31'].copy()
data['DATE'] = pd.to_datetime(data.index)


def make_idata_filename(model_name: str, period_label: str) -> str:
    parts = model_name.lower().split('_')
    infl = parts[0]
    family = parts[1]
    if parts[2] == 'orth':
        corr = 'uncorr'
        gap = '_'.join(parts[3:])
    else:
        corr = 'corr'
        gap = '_'.join(parts[2:])
    return f"{infl}_{family}_{gap}_{period_label}_{corr}"


gibbs_module = importlib.reload(gibbs_module)
run_hsa_dynamic = gibbs_module.run_hsa_dynamic
draws_to_idata = gibbs_module.draws_to_idata

dict_idata = {}
dict_draws = {}
print(f'=== Run Gibbs HSA dynamic models ({n_name}, {period_title}) ===')

for infl, spec in inflation_specs.items():
    print(f'--- Inflation: {infl} ---')

    for x_name, x_col in x_specs.items():
        sample = prepare_gibbs_sample(
            data,
            pi_col=spec['pi'],
            pi_prev_col=spec['pi_prev'],
            pi_expect_col=spec['pi_expect'],
            x_col=x_col,
            x_prev_col=f'{x_col}_prev',
            n_col=n_col,
        )
        print(f'    Sample {x_name}: {format_sample_window(sample)}')

        run_name = f"{infl.upper()}_HSA_{x_name}"
        orth_run_name = f"{infl.upper()}_HSA_orth_{x_name}"
        print(f'    Running Gibbs model: {run_name}')
        draws = run_hsa_dynamic(
            pi=np.asarray(sample['pi'], dtype=float),
            pi_prev=np.asarray(sample['pi_prev'], dtype=float),
            pi_expect=np.asarray(sample['pi_expect'], dtype=float),
            x=np.asarray(sample['x'], dtype=float),
            x_prev=np.asarray(sample['x_prev'], dtype=float),
            N=np.asarray(sample['N'], dtype=float),
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=False,
        )
        dict_draws[run_name] = draws
        dict_idata[run_name] = draws_to_idata(draws)

        print(f'    Running Gibbs model: {orth_run_name}')
        orth_draws = run_hsa_dynamic(
            pi=np.asarray(sample['pi'], dtype=float),
            pi_prev=np.asarray(sample['pi_prev'], dtype=float),
            pi_expect=np.asarray(sample['pi_expect'], dtype=float),
            x=np.asarray(sample['x'], dtype=float),
            x_prev=np.asarray(sample['x_prev'], dtype=float),
            N=np.asarray(sample['N'], dtype=float),
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=True,
        )
        dict_draws[orth_run_name] = orth_draws
        dict_idata[orth_run_name] = draws_to_idata(orth_draws)

print('=== All Gibbs HSA dynamic models finished ===')

base_models_to_show = [
    'HSA_unemp_gap',
    'HSA_output_gap_BN',
    'HSA_markup_BN_inv',
    'HSA_markup_inv',
]

for infl in inflation_specs.keys():
    models_to_show = [f"{infl.upper()}_{m}" for m in base_models_to_show]
    orth_models_to_show = [m.replace('_HSA_', '_HSA_orth_') for m in models_to_show]
    all_models_to_show = models_to_show + orth_models_to_show
    dict_items_fill = {k: dict_idata[k] for k in all_models_to_show if k in dict_idata}

    display_hmc_results(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        tex_dir=tex_dir / period_label / infl.lower(),
        params=('alpha', 'kappa', 'theta', 'phi_1', 'rho_1', 'rho_2', 'rho', 'sigma_e', 'sigma_zeta', 'sigma_u', 'sigma_eps'),
        title=f'Gibbs HSA Dynamic Results ({infl.upper()})',
    )

    fig_dir = base_fig_dir / 'gibbs_hsa_dynamic' / period_label / infl.lower()
    fig_dir.mkdir(parents=True, exist_ok=True)

    display_hmc_posterior_prior(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        fig_dir=fig_dir,
        params=('alpha', 'kappa', 'theta', 'phi_1', 'rho_1', 'rho_2'),
        title=f'Prior vs Posterior ({infl.upper()})',
    )

    idata_dir = idat_dir / f'gibbs_hsa_dynamic_{period_label}_{infl.lower()}'
    dict_items_save = {make_idata_filename(k, period_label): v for k, v in dict_items_fill.items()}
    save_idata_map(dict_items_save, idata_dir)


=== Run Gibbs HSA dynamic models (gustavo, 1982-2012) ===
--- Inflation: ppi ---
    Sample unemp_gap: 1982Q1-2012Q4 (T=124)
    Running Gibbs model: PPI_HSA_unemp_gap
    Running Gibbs model: PPI_HSA_orth_unemp_gap
    Sample output_gap_BN: 1982Q1-2012Q4 (T=124)
    Running Gibbs model: PPI_HSA_output_gap_BN
    Running Gibbs model: PPI_HSA_orth_output_gap_BN
    Sample markup_BN_inv: 1982Q1-2012Q4 (T=124)
    Running Gibbs model: PPI_HSA_markup_BN_inv
    Running Gibbs model: PPI_HSA_orth_markup_BN_inv
    Sample markup_inv: 1982Q1-2012Q4 (T=124)
    Running Gibbs model: PPI_HSA_markup_inv
    Running Gibbs model: PPI_HSA_orth_markup_inv
--- Inflation: cpi ---
    Sample unemp_gap: 1982Q1-2012Q4 (T=124)
    Running Gibbs model: CPI_HSA_unemp_gap
    Running Gibbs model: CPI_HSA_orth_unemp_gap
    Sample output_gap_BN: 1982Q1-2012Q4 (T=124)
    Running Gibbs model: CPI_HSA_output_gap_BN
    Running Gibbs model: CPI_HSA_orth_output_gap_BN
    Sample markup_BN_inv: 1982Q1-2012Q4 (T=124)

In [5]:
period_label = '1988_2017'
period_title = '1988-2017'
n_name = 'tnic'
n_col = 'N_TNIC'
data = data_origin.loc['1988-03-31':'2017-12-31'].copy()
data['DATE'] = pd.to_datetime(data.index)

gibbs_module = importlib.reload(gibbs_module)
run_hsa_dynamic = gibbs_module.run_hsa_dynamic
draws_to_idata = gibbs_module.draws_to_idata

dict_idata = {}
dict_draws = {}
print(f'=== Run Gibbs HSA dynamic models ({n_name}, {period_title}) ===')

for infl, spec in inflation_specs.items():
    print(f'--- Inflation: {infl} ---')

    for x_name, x_col in x_specs.items():
        sample = prepare_gibbs_sample(
            data,
            pi_col=spec['pi'],
            pi_prev_col=spec['pi_prev'],
            pi_expect_col=spec['pi_expect'],
            x_col=x_col,
            x_prev_col=f'{x_col}_prev',
            n_col=n_col,
        )
        print(f'    Sample {x_name}: {format_sample_window(sample)}')

        run_name = f"{infl.upper()}_HSA_{x_name}"
        orth_run_name = f"{infl.upper()}_HSA_orth_{x_name}"
        print(f'    Running Gibbs model: {run_name}')
        draws = run_hsa_dynamic(
            pi=np.asarray(sample['pi'], dtype=float),
            pi_prev=np.asarray(sample['pi_prev'], dtype=float),
            pi_expect=np.asarray(sample['pi_expect'], dtype=float),
            x=np.asarray(sample['x'], dtype=float),
            x_prev=np.asarray(sample['x_prev'], dtype=float),
            N=np.asarray(sample['N'], dtype=float),
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=False,
        )
        dict_draws[run_name] = draws
        dict_idata[run_name] = draws_to_idata(draws)

        print(f'    Running Gibbs model: {orth_run_name}')
        orth_draws = run_hsa_dynamic(
            pi=np.asarray(sample['pi'], dtype=float),
            pi_prev=np.asarray(sample['pi_prev'], dtype=float),
            pi_expect=np.asarray(sample['pi_expect'], dtype=float),
            x=np.asarray(sample['x'], dtype=float),
            x_prev=np.asarray(sample['x_prev'], dtype=float),
            N=np.asarray(sample['N'], dtype=float),
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=True,
        )
        dict_draws[orth_run_name] = orth_draws
        dict_idata[orth_run_name] = draws_to_idata(orth_draws)

print('=== All Gibbs HSA dynamic models finished ===')

base_models_to_show = [
    'HSA_unemp_gap',
    'HSA_output_gap_BN',
    'HSA_markup_BN_inv',
    'HSA_markup_inv',
]

for infl in inflation_specs.keys():
    models_to_show = [f"{infl.upper()}_{m}" for m in base_models_to_show]
    orth_models_to_show = [m.replace('_HSA_', '_HSA_orth_') for m in models_to_show]
    all_models_to_show = models_to_show + orth_models_to_show
    dict_items_fill = {k: dict_idata[k] for k in all_models_to_show if k in dict_idata}

    display_hmc_results(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        tex_dir=tex_dir / period_label / infl.lower(),
        params=('alpha', 'kappa', 'theta', 'phi_1', 'rho_1', 'rho_2', 'rho', 'sigma_e', 'sigma_zeta', 'sigma_u', 'sigma_eps'),
        title=f'Gibbs HSA Dynamic Results ({infl.upper()})',
    )

    fig_dir = base_fig_dir / 'gibbs_hsa_dynamic' / period_label / infl.lower()
    fig_dir.mkdir(parents=True, exist_ok=True)

    display_hmc_posterior_prior(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        fig_dir=fig_dir,
        params=('alpha', 'kappa', 'theta', 'phi_1', 'rho_1', 'rho_2'),
        title=f'Prior vs Posterior ({infl.upper()})',
    )

    idata_dir = idat_dir / f'gibbs_hsa_dynamic_{period_label}_{infl.lower()}'
    dict_items_save = {make_idata_filename(k, period_label): v for k, v in dict_items_fill.items()}
    save_idata_map(dict_items_save, idata_dir)


=== Run Gibbs HSA dynamic models (tnic, 1988-2017) ===
--- Inflation: ppi ---
    Sample unemp_gap: 1988Q1-2017Q4 (T=120)
    Running Gibbs model: PPI_HSA_unemp_gap
    Running Gibbs model: PPI_HSA_orth_unemp_gap
    Sample output_gap_BN: 1988Q1-2017Q4 (T=120)
    Running Gibbs model: PPI_HSA_output_gap_BN
    Running Gibbs model: PPI_HSA_orth_output_gap_BN
    Sample markup_BN_inv: 1988Q1-2017Q4 (T=120)
    Running Gibbs model: PPI_HSA_markup_BN_inv
    Running Gibbs model: PPI_HSA_orth_markup_BN_inv
    Sample markup_inv: 1988Q1-2017Q4 (T=120)
    Running Gibbs model: PPI_HSA_markup_inv
    Running Gibbs model: PPI_HSA_orth_markup_inv
--- Inflation: cpi ---
    Sample unemp_gap: 1988Q1-2017Q4 (T=120)
    Running Gibbs model: CPI_HSA_unemp_gap
    Running Gibbs model: CPI_HSA_orth_unemp_gap
    Sample output_gap_BN: 1988Q1-2017Q4 (T=120)
    Running Gibbs model: CPI_HSA_output_gap_BN
    Running Gibbs model: CPI_HSA_orth_output_gap_BN
    Sample markup_BN_inv: 1988Q1-2017Q4 (T=120)
  